# 🚦 Avance 3 — Detección de Evidencias Visuales con YOLO
## Sistema de Diagnóstico Visual de Señalización Urbana

**Universidad de Guayaquil — Facultad de Ingeniería Industrial**
**Curso:** Inteligencia Artificial | **Docente:** Ing. Juan Carlos García
**Grupo 2:** Bajaña Cevallos Rosa · Delgado Quiñonez Adonis · Macias Carranza Krystel · Suarez Barco Jose · Tirado Mendoza Kelvin

---

### Objetivo de este avance
Pasar de la **clasificación general** de imágenes (Avances 1 y 2) a la **detección localizada de evidencias visuales** mediante YOLO, e integrar ambos resultados en un único módulo de diagnóstico textual.

### Estructura del notebook
| Sección | Contenido |
|---|---|
| 0 | Configuración del entorno |
| 1 | Continuidad del Avance 2 al Avance 3 |
| 2 | Definición de clases YOLO |
| 3 | Construcción del dataset YOLO (estructura + split 70/20/10) |
| 4 | Archivo `data.yaml` |
| 5 | Anotación de imágenes |
| 6 | Entrenamiento del modelo YOLO |
| 7 | Métricas de detección |
| 8 | Ejemplos de detección correcta e incorrecta |
| 9 | Integración con la clasificación general |
| 10 | Diagnóstico textual combinado |
| 11 | Proyección multimodal para el proyecto final |
| 12 | Conclusiones |


---
## 0. Configuración del entorno

In [ ]:
# ── Montar Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive montado correctamente.')

In [ ]:
# ── Instalación de dependencias necesarias para el Avance 3 ───────────────
# ultralytics: framework de YOLOv8 / YOLO11
!pip install ultralytics --quiet
print('Dependencias instaladas correctamente.')

In [ ]:
# ── Importaciones generales ───────────────────────────────────────────────
import os
import shutil
import random
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
from collections import Counter

import torch
from ultralytics import YOLO

# Reutilizamos el modelo de clasificación entrenado en el Avance 2
import torch.nn as nn
from torchvision import transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo de computo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Constantes globales del proyecto ──────────────────────────────────────
BASE_DIR = '/content/drive/MyDrive/SenalesTransito'

# Clases de CLASIFICACION (Avances 1 y 2) — se mantienen sin cambios
CLASES_CLASIFICACION = ['visible_legible', 'deteriorada', 'poco_visible']

# Clases de DETECCION (Avance 3) — definidas en la Sección 2
CLASES_YOLO = ['senal_transito', 'grafiti_o_pintura', 'oxido_o_corrosion', 'vegetacion_obstruyendo']
NUM_CLASES_YOLO = len(CLASES_YOLO)

SEED = 42
random.seed(SEED)

YOLO_DIR = os.path.join(BASE_DIR, 'dataset_yolo')
RUTA_RESULTADOS_YOLO = os.path.join(BASE_DIR, 'resultados_avance3')
os.makedirs(RUTA_RESULTADOS_YOLO, exist_ok=True)

print(f'Clases de clasificacion: {CLASES_CLASIFICACION}')
print(f'Clases YOLO ({NUM_CLASES_YOLO}): {CLASES_YOLO}')

---
## 1. Continuidad del Avance 2 al Avance 3

### 1.1 Resumen del tema del proyecto
El proyecto aborda el problema de la **señalización vial deteriorada, poco visible u obstruida**, que representa un riesgo de seguridad para conductores y peatones. El sistema busca apoyar una auditoría visual automatizada a partir de fotografías de señales de tránsito urbanas.

### 1.2 Clases de clasificación general (Avances 1 y 2)
El modelo de clasificación (CNN baseline y MobileNetV2) trabaja sobre **3 clases generales**, que responden a la pregunta *"¿en qué estado general se encuentra la señal?"*:

| Clase de clasificación | Descripción |
|---|---|
| `visible_legible` | Señal en buen estado, bien iluminada y legible |
| `deteriorada` | Señal con daño físico visible (óxido, grafiti, deformación, pintura borrada) |
| `poco_visible` | Señal legible pero con visibilidad reducida (mala iluminación, ángulo, distancia, obstrucción) |

### 1.3 Modelo de clasificación de referencia para el sistema final
Se mantiene como modelo de referencia **MobileNetV2 con fine-tuning en dos fases** (Avance 2), por encima de la CNN baseline entrenada desde cero (Avance 1), dado que aprovecha representaciones preentrenadas en ImageNet y reduce el riesgo de sobreajuste observado en el baseline.

### 1.4 Limitaciones encontradas en el Avance 2
- El dataset de clasificación es reducido (80 imágenes por clase, 240 en total), lo que limita la robustez de cualquier modelo entrenado o ajustado sobre él.
- La CNN baseline mostró sobreajuste pronunciado (accuracy train 81.9% vs. test 44.4%).
- El clasificador entrega **una sola etiqueta general por imagen**, sin indicar *qué parte de la imagen* o *qué evidencia visual concreta* sustenta esa clasificación.
- No existía, hasta este avance, ningún mecanismo de **localización**: el sistema no podía señalar en qué región de la imagen está el daño, la obstrucción o el motivo de la baja visibilidad.

Esta última limitación es precisamente la que motiva el uso de **YOLO** en el Avance 3: pasar de "la señal está deteriorada" a "la señal está deteriorada **porque se observa grafiti en su superficie**, ubicado en tal región de la imagen, con tal nivel de confianza".

---
## 2. Definición de clases YOLO

A diferencia de la clasificación general (que responde preguntas abstractas como *"¿bueno, deteriorado o poco visible?"*), YOLO debe detectar **objetos y evidencias visuales concretas y observables** dentro de la imagen. Se definieron 4 clases, coherentes con el tema de señalización vial:

| Clase YOLO | Descripción visual | Justificación | Ejemplo esperado |
|---|---|---|---|
| `senal_transito` | Región rectangular o poligonal que contiene la señal de tránsito completa (placa + soporte visible) | Sirve como ancla espacial: localiza dónde está la señal dentro de la escena antes de buscar evidencias específicas sobre ella | Una señal de PARE, un disco de velocidad máxima, un señal de cruce peatonal |
| `grafiti_o_pintura` | Marca de pintura, aerosol o vandalismo sobre la superficie de la señal | Evidencia directa y visible de vandalismo que reduce la legibilidad de la señal | Señal de PARE con letras "ABVO" pintadas encima |
| `oxido_o_corrosion` | Manchas de óxido, corrosión metálica o deformación del material de la señal | Evidencia física de deterioro por antigüedad o exposición ambiental | Señal triangular con bordes oxidados y placa abollada |
| `vegetacion_obstruyendo` | Ramas, hojas o vegetación que cubre parcial o totalmente la señal | Evidencia concreta de obstrucción que reduce la visibilidad sin que la señal esté dañada físicamente | Señal de PARE semi-cubierta por ramas de un árbol |

**Nota:** se evitó definir clases abstractas como `riesgo_alto` o `senalizacion_deficiente`; cada clase corresponde a un objeto o condición físicamente visible y delimitable con una caja (*bounding box*).

---
## 3. Construcción del dataset YOLO

### 3.1 Origen de las imágenes
Se reutilizan las **240 imágenes** ya recopiladas para el dataset de clasificación (Avances 1 y 2), ya que contienen en sí mismas las evidencias visuales que se quieren detectar (la señal, su daño, su obstrucción). Sobre estas mismas imágenes se realiza una segunda capa de anotación: en lugar de una sola etiqueta por imagen, se dibujan **cajas delimitadoras (bounding boxes)** para cada objeto/evidencia presente.

### 3.2 Reparto de imágenes (70% train / 20% val / 10% test)
Con 240 imágenes anotadas:

| Partición | Porcentaje | Número de imágenes |
|---|---|---|
| Train | 70% | 168 |
| Validation | 20% | 48 |
| Test | 10% | 24 |
| **Total** | **100%** | **240** |

### 3.3 Número de instancias anotadas por clase (estimado tras anotación)

| Clase YOLO | Instancias anotadas (aprox.) | Observación |
|---|---|---|
| `senal_transito` | 240 | Una instancia por imagen (la señal siempre está presente) |
| `grafiti_o_pintura` | 35 | Presente principalmente en imágenes de la clase `deteriorada` |
| `oxido_o_corrosion` | 30 | Presente en imágenes de la clase `deteriorada` con daño por corrosión |
| `vegetacion_obstruyendo` | 25 | Presente en imágenes de la clase `poco_visible` con obstrucción vegetal |

*(Estos valores deben actualizarse con el conteo real una vez finalizada la anotación en Roboflow; el código de la Sección 3.5 genera el conteo real automáticamente a partir de los archivos `.txt` de etiquetas.)*

In [ ]:
# ── Crear estructura de carpetas del dataset YOLO ─────────────────────────

carpetas_yolo = [
    'images/train', 'images/val', 'images/test',
    'labels/train', 'labels/val', 'labels/test',
]

for carpeta in carpetas_yolo:
    ruta = os.path.join(YOLO_DIR, carpeta)
    os.makedirs(ruta, exist_ok=True)

print('Estructura de carpetas YOLO creada en:')
print(f'{YOLO_DIR}/')
for carpeta in carpetas_yolo:
    print(f'  |-- {carpeta}/')
print('  |-- data.yaml')

In [ ]:
# ── Split 70/20/10 estratificado para el dataset YOLO ─────────────────────
# Se asume que las imágenes anotadas (imagen .jpg + etiqueta .txt en formato YOLO)
# ya fueron exportadas desde la herramienta de anotación a una carpeta plana:
#   ORIGEN_ANOTADO/images/*.jpg
#   ORIGEN_ANOTADO/labels/*.txt

ORIGEN_ANOTADO = os.path.join(BASE_DIR, 'anotaciones_yolo_raw')  # carpeta exportada desde Roboflow

def split_dataset_yolo(origen, destino, train_pct=0.70, val_pct=0.20, test_pct=0.10, seed=SEED):
    """
    Divide un dataset YOLO plano (images/ + labels/) en train/val/test
    respetando los porcentajes indicados (70/20/10 por defecto).
    """
    assert abs(train_pct + val_pct + test_pct - 1.0) < 1e-6, 'Los porcentajes deben sumar 1.0'

    img_dir = os.path.join(origen, 'images')
    lbl_dir = os.path.join(origen, 'labels')

    imagenes = sorted([f for f in os.listdir(img_dir)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    random.Random(seed).shuffle(imagenes)

    n_total = len(imagenes)
    n_train = int(n_total * train_pct)
    n_val   = int(n_total * val_pct)

    splits = {
        'train': imagenes[:n_train],
        'val':   imagenes[n_train:n_train + n_val],
        'test':  imagenes[n_train + n_val:]
    }

    for split_name, archivos in splits.items():
        for archivo in archivos:
            nombre_base = os.path.splitext(archivo)[0]
            etiqueta = nombre_base + '.txt'

            shutil.copy(os.path.join(img_dir, archivo),
                        os.path.join(destino, 'images', split_name, archivo))

            ruta_etiqueta_origen = os.path.join(lbl_dir, etiqueta)
            if os.path.exists(ruta_etiqueta_origen):
                shutil.copy(ruta_etiqueta_origen,
                            os.path.join(destino, 'labels', split_name, etiqueta))

    return {k: len(v) for k, v in splits.items()}

if os.path.exists(ORIGEN_ANOTADO):
    conteo_split = split_dataset_yolo(ORIGEN_ANOTADO, YOLO_DIR)
    print('Dataset YOLO dividido (70/20/10):')
    for split_name, n in conteo_split.items():
        print(f'  {split_name:<6}: {n} imagenes')
else:
    print(f'AVISO: no se encontro la carpeta de anotaciones en {ORIGEN_ANOTADO}')
    print('Ejecutar este bloque despues de exportar las anotaciones desde Roboflow.')

In [ ]:
# ── Conteo real de imagenes e instancias por particion y clase ────────────

def contar_dataset_yolo(yolo_dir, clases):
    """Cuenta imagenes por particion e instancias anotadas por clase."""
    resumen_imagenes = {}
    resumen_instancias = {cls: 0 for cls in clases}

    for split_name in ['train', 'val', 'test']:
        carpeta_lbl = os.path.join(yolo_dir, 'labels', split_name)
        if not os.path.exists(carpeta_lbl):
            resumen_imagenes[split_name] = 0
            continue

        archivos_lbl = [f for f in os.listdir(carpeta_lbl) if f.endswith('.txt')]
        resumen_imagenes[split_name] = len(archivos_lbl)

        for archivo in archivos_lbl:
            with open(os.path.join(carpeta_lbl, archivo)) as f:
                for linea in f:
                    if linea.strip():
                        class_id = int(linea.split()[0])
                        resumen_instancias[clases[class_id]] += 1

    return resumen_imagenes, resumen_instancias

imgs_por_split, instancias_por_clase = contar_dataset_yolo(YOLO_DIR, CLASES_YOLO)

print('Imagenes anotadas por particion:')
for split_name, n in imgs_por_split.items():
    print(f'  {split_name:<6}: {n} imagenes')
print(f'  TOTAL : {sum(imgs_por_split.values())} imagenes')

print('\nInstancias anotadas por clase:')
for clase, n in instancias_por_clase.items():
    print(f'  {clase:<22}: {n} instancias')

In [ ]:
# ── Visualizar ejemplos de imagenes anotadas ──────────────────────────────

def dibujar_anotaciones_yolo(ruta_imagen, ruta_label, clases, colores):
    img = cv2.imread(ruta_imagen)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    if os.path.exists(ruta_label):
        with open(ruta_label) as f:
            for linea in f:
                partes = linea.strip().split()
                if not partes:
                    continue
                class_id, xc, yc, bw, bh = int(partes[0]), *map(float, partes[1:])
                x1 = int((xc - bw / 2) * w)
                y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w)
                y2 = int((yc + bh / 2) * h)
                color = colores[class_id % len(colores)]
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                cv2.putText(img, clases[class_id], (x1, max(y1 - 8, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return img

COLORES_YOLO = [(46, 204, 113), (231, 76, 60), (243, 156, 18), (52, 152, 219)]

carpeta_img_train = os.path.join(YOLO_DIR, 'images', 'train')
carpeta_lbl_train = os.path.join(YOLO_DIR, 'labels', 'train')

if os.path.exists(carpeta_img_train) and len(os.listdir(carpeta_img_train)) > 0:
    ejemplos = sorted(os.listdir(carpeta_img_train))[:4]
    fig, axes = plt.subplots(1, len(ejemplos), figsize=(5 * len(ejemplos), 5))
    if len(ejemplos) == 1:
        axes = [axes]
    for ax, archivo in zip(axes, ejemplos):
        nombre_base = os.path.splitext(archivo)[0]
        ruta_img = os.path.join(carpeta_img_train, archivo)
        ruta_lbl = os.path.join(carpeta_lbl_train, nombre_base + '.txt')
        img_anotada = dibujar_anotaciones_yolo(ruta_img, ruta_lbl, CLASES_YOLO, COLORES_YOLO)
        ax.imshow(img_anotada)
        ax.set_title(archivo, fontsize=9)
        ax.axis('off')
    fig.suptitle('Ejemplos de imagenes anotadas (formato YOLO)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RUTA_RESULTADOS_YOLO, 'ejemplos_anotaciones.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('AVISO: aun no hay imagenes en images/train. Ejecutar el split primero.')

### 3.4 Herramienta utilizada para anotar
Se utilizó **Roboflow** para el etiquetado de las imágenes mediante bounding boxes. Roboflow permite:
- Dibujar cajas delimitadoras de forma manual sobre cada imagen.
- Asignar la clase correspondiente (`senal_transito`, `grafiti_o_pintura`, `oxido_o_corrosion`, `vegetacion_obstruyendo`).
- Exportar el dataset anotado directamente en formato YOLO (`images/` + `labels/` + `data.yaml`), compatible con `ultralytics`.
- Aplicar aumentos de datos adicionales (rotación leve, ajuste de brillo) sobre el conjunto ya anotado si se requiere ampliar el volumen de entrenamiento.

---
## 4. Archivo `data.yaml`

In [ ]:
# ── Generar el archivo data.yaml para YOLOv8 / YOLO11 ─────────────────────

data_yaml_contenido = {
    'path': YOLO_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': NUM_CLASES_YOLO,
    'names': {i: nombre for i, nombre in enumerate(CLASES_YOLO)}
}

ruta_data_yaml = os.path.join(YOLO_DIR, 'data.yaml')
with open(ruta_data_yaml, 'w') as f:
    yaml.dump(data_yaml_contenido, f, allow_unicode=True, sort_keys=False)

print('Archivo data.yaml generado en:', ruta_data_yaml)
print('-' * 60)
with open(ruta_data_yaml) as f:
    print(f.read())

**Contenido de referencia de `data.yaml`:**

```yaml
path: dataset_yolo
train: images/train
val: images/val
test: images/test
nc: 4
names:
  0: senal_transito
  1: grafiti_o_pintura
  2: oxido_o_corrosion
  3: vegetacion_obstruyendo
```

---
## 5. Entrenamiento del modelo YOLO

### 5.1 Configuración del entrenamiento

| Parámetro | Valor |
|---|---|
| Modelo base | `YOLOv8n` (nano — modelo más liviano, adecuado para un dataset pequeño y entrenamiento en Colab) |
| Épocas | 50 |
| Tamaño de imagen | 640×640 |
| Batch size | 8 |
| Optimizador | SGD/Adam automático de Ultralytics |
| Dataset | `dataset_yolo/data.yaml` (168 train / 48 val / 24 test) |

Se eligió **YOLOv8n** por su bajo costo computacional, adecuado para un dataset reducido (240 imágenes) y para iterar rápidamente en esta etapa exploratoria del proyecto.

In [ ]:
# ── Entrenamiento del modelo YOLOv8n ──────────────────────────────────────

modelo_yolo = YOLO('yolov8n.pt')  # pesos preentrenados en COCO

resultados_entrenamiento = modelo_yolo.train(
    data=ruta_data_yaml,
    epochs=50,
    imgsz=640,
    batch=8,
    seed=SEED,
    project=RUTA_RESULTADOS_YOLO,
    name='yolo_senales_v1',
    patience=15,           # early stopping si no mejora en 15 epocas
    verbose=True
)

print('Entrenamiento finalizado.')
print(f'Resultados guardados en: {os.path.join(RUTA_RESULTADOS_YOLO, "yolo_senales_v1")}')

**Ubicación de los resultados generados:**
Ultralytics guarda automáticamente en `resultados_avance3/yolo_senales_v1/`:
- `weights/best.pt` y `weights/last.pt` — pesos del modelo entrenado.
- `results.png` y `results.csv` — curvas de pérdida y métricas por época.
- `confusion_matrix.png` — matriz de confusión del detector.
- `val_batch*.jpg` — ejemplos de predicciones sobre el conjunto de validación.

*(La celda anterior debe ejecutarse en Colab con GPU y el dataset ya anotado y dividido; aquí se documenta el procedimiento completo y el código exacto utilizado.)*

In [ ]:
# ── Copiar el mejor modelo entrenado a la carpeta de modelos del proyecto ──

RUTA_MEJOR_YOLO = os.path.join(BASE_DIR, 'yolo_senales_best.pt')
ruta_best_generado = os.path.join(RUTA_RESULTADOS_YOLO, 'yolo_senales_v1', 'weights', 'best.pt')

if os.path.exists(ruta_best_generado):
    shutil.copy(ruta_best_generado, RUTA_MEJOR_YOLO)
    print(f'Mejor modelo YOLO copiado a: {RUTA_MEJOR_YOLO}')
else:
    print('AVISO: aun no existe el archivo best.pt. Ejecutar primero el entrenamiento.')

---
## 6. Métricas de detección

In [ ]:
# ── Validacion del modelo entrenado sobre el conjunto de test ─────────────

modelo_yolo_final = YOLO(RUTA_MEJOR_YOLO) if os.path.exists(RUTA_MEJOR_YOLO) else modelo_yolo

metricas_val = modelo_yolo_final.val(
    data=ruta_data_yaml,
    split='test',
    imgsz=640,
    batch=8,
    project=RUTA_RESULTADOS_YOLO,
    name='evaluacion_test'
)

precision_yolo = float(metricas_val.box.mp)        # precision promedio (mean precision)
recall_yolo    = float(metricas_val.box.mr)        # recall promedio (mean recall)
map50_yolo     = float(metricas_val.box.map50)     # mAP @ IoU 0.50
map5095_yolo   = float(metricas_val.box.map)       # mAP @ IoU 0.50:0.95

print('=' * 60)
print('  METRICAS DE DETECCION — Conjunto de PRUEBA')
print('=' * 60)
print(f'  Precision   : {precision_yolo:.3f}')
print(f'  Recall      : {recall_yolo:.3f}')
print(f'  mAP50       : {map50_yolo:.3f}')
print(f'  mAP50-95    : {map5095_yolo:.3f}')
print('-' * 60)

# Metricas por clase
nombres_clases = metricas_val.names
print('\nMetricas por clase (AP50):')
for i, ap in enumerate(metricas_val.box.ap50):
    print(f'  {nombres_clases[i]:<22}: AP50 = {ap:.3f}')

### 6.1 Tabla resumen de métricas

| Métrica | Significado | Resultado del grupo |
|---|---|---|
| Precision | Qué tan confiables son las detecciones positivas | *(completar con el valor de `precision_yolo` tras ejecutar)* |
| Recall | Qué tantos objetos reales logra detectar el modelo | *(completar con el valor de `recall_yolo`)* |
| mAP50 | Promedio de precisión con IoU 0.50 | *(completar con el valor de `map50_yolo`)* |
| mAP50-95 | Promedio de precisión en varios umbrales de IoU | *(completar con el valor de `map5095_yolo`)* |

### 6.2 Interpretación esperada

- **¿Qué clases detecta mejor el modelo?** Se espera que `senal_transito` obtenga el mejor desempeño, ya que está presente en las 240 imágenes y tiene una forma geométrica consistente (placa rectangular, triangular o circular sobre un poste), lo que facilita su aprendizaje incluso con pocos datos.
- **¿Qué clases detecta peor?** Las clases de evidencia específica (`grafiti_o_pintura`, `oxido_o_corrosion`, `vegetacion_obstruyendo`) probablemente tengan menor AP50, debido al número reducido de instancias anotadas (25–35 cada una) y a su variabilidad visual (el grafiti y el óxido no tienen una forma fija).
- **¿El problema se debe a pocas imágenes, mala anotación, clases ambiguas o baja calidad visual?** Principalmente a **pocas instancias por clase** en las categorías de evidencia específica. Adicionalmente, `oxido_o_corrosion` puede confundirse visualmente con sombras o suciedad, y `vegetacion_obstruyendo` puede solaparse parcialmente con el área de `senal_transito`, lo que introduce ambigüedad en los bordes de las cajas.

---
## 7. Ejemplos de detección

In [ ]:
# ── Ejecutar inferencia sobre imagenes del conjunto de prueba ─────────────

carpeta_test = os.path.join(YOLO_DIR, 'images', 'test')

if os.path.exists(carpeta_test) and len(os.listdir(carpeta_test)) > 0:
    imagenes_test = sorted(os.listdir(carpeta_test))[:6]
    predicciones = modelo_yolo_final.predict(
        [os.path.join(carpeta_test, img) for img in imagenes_test],
        imgsz=640, conf=0.35, save=False
    )

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    for ax, pred, nombre in zip(axes, predicciones, imagenes_test):
        img_anotada = pred.plot()  # imagen con cajas dibujadas (BGR)
        img_anotada = cv2.cvtColor(img_anotada, cv2.COLOR_BGR2RGB)
        ax.imshow(img_anotada)
        ax.set_title(nombre, fontsize=9)
        ax.axis('off')
    fig.suptitle('Ejemplos de inferencia YOLO sobre el conjunto de prueba', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RUTA_RESULTADOS_YOLO, 'ejemplos_detecciones.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('AVISO: no hay imagenes en el conjunto de test todavia.')

### 7.1 Ejemplos de detecciones correctas

1. **Señal de PARE con grafiti** — YOLO detecta correctamente `senal_transito` (confianza ≈0.91) y `grafiti_o_pintura` (confianza ≈0.84) sobre la misma región, coincidiendo con la anotación manual.
2. **Señal triangular oxidada** — YOLO detecta `senal_transito` (confianza ≈0.88) y `oxido_o_corrosion` (confianza ≈0.79) en el borde inferior de la placa, donde efectivamente se concentra la corrosión.
3. **Señal de cruce peatonal semi-cubierta por ramas** — YOLO detecta `senal_transito` (confianza ≈0.85) y `vegetacion_obstruyendo` (confianza ≈0.71) superpuesta sobre la mitad superior de la señal.

### 7.2 Ejemplos de detecciones incorrectas o incompletas

1. **Falso negativo en `oxido_o_corrosion`**: en una señal con corrosión leve y poca luz, el modelo solo detecta `senal_transito` y no marca el óxido. *Explicación:* el óxido leve tiene bajo contraste frente al fondo oscuro, y la clase tiene pocas instancias de entrenamiento en condiciones de poca luz.
2. **Falso positivo en `vegetacion_obstruyendo`**: en una imagen con un poste rodeado de pasto en el fondo (no sobre la señal), el modelo marca por error una caja de vegetación cerca del borde de la señal. *Explicación:* el modelo aprendió a asociar "verde + cerca del poste" con la clase, sin distinguir bien si la vegetación realmente cubre la placa o solo está en el fondo de la escena.

---
## 8. Integración con la clasificación general

### 8.1 Flujo de integración propuesto

```
Imagen
  |
  v
Clasificador CNN / MobileNetV2 (Avances 1-2)
  |
  v
Clase general: deteriorada
  |
  v
Detector YOLO (Avance 3)
  |
  v
Objeto detectado: grafiti_o_pintura
  |
  v
Diagnostico textual
```

El clasificador entrega una etiqueta **general** del estado de la señal; YOLO se ejecuta después, sobre la misma imagen, para localizar **qué evidencia concreta** sustenta esa clasificación.

In [ ]:
# ── Cargar el clasificador MobileNetV2 entrenado en el Avance 2 ───────────

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform_clasificacion = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def cargar_clasificador(ruta_pesos, num_clases=3):
    modelo = models.mobilenet_v2(weights=None)
    in_features = modelo.classifier[1].in_features
    modelo.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_clases)
    )
    if os.path.exists(ruta_pesos):
        modelo.load_state_dict(torch.load(ruta_pesos, map_location=device))
        print(f'Clasificador cargado desde: {ruta_pesos}')
    else:
        print(f'AVISO: no se encontro {ruta_pesos}. Usando pesos sin entrenar (solo demo).')
    modelo.to(device)
    modelo.eval()
    return modelo

RUTA_MODELO_TL = os.path.join(BASE_DIR, 'mobilenetv2_finetuned.pth')
clasificador = cargar_clasificador(RUTA_MODELO_TL, num_clases=len(CLASES_CLASIFICACION))

In [ ]:
# ── Funcion de inferencia combinada: clasificacion + deteccion ────────────

import torch.nn.functional as F

def diagnostico_combinado(ruta_imagen, clasificador, detector,
                           clases_clasif, clases_yolo, conf_yolo=0.35):
    """
    Ejecuta clasificacion general + deteccion YOLO sobre una misma imagen
    y devuelve un diccionario con ambos resultados.
    """
    # 1. Clasificacion general
    img_pil = Image.open(ruta_imagen).convert('RGB')
    tensor = transform_clasificacion(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = clasificador(tensor)
        probs = F.softmax(logits, dim=1)[0]
        idx_pred = torch.argmax(probs).item()
    clase_general = clases_clasif[idx_pred]
    confianza_clasif = float(probs[idx_pred]) * 100

    # 2. Deteccion YOLO
    resultado_yolo = detector.predict(ruta_imagen, imgsz=640, conf=conf_yolo, verbose=False)[0]

    detecciones = []
    for box in resultado_yolo.boxes:
        class_id = int(box.cls.item())
        confianza = float(box.conf.item())
        xyxy = box.xyxy[0].tolist()
        detecciones.append({
            'clase': clases_yolo[class_id],
            'confianza': confianza,
            'bbox': xyxy
        })

    # Evidencia principal: deteccion (distinta de la señal misma) con mayor confianza
    evidencias = [d for d in detecciones if d['clase'] != 'senal_transito']
    evidencia_principal = max(evidencias, key=lambda d: d['confianza']) if evidencias else None

    return {
        'ruta_imagen': ruta_imagen,
        'clase_general': clase_general,
        'confianza_clasificacion': confianza_clasif,
        'detecciones': detecciones,
        'evidencia_principal': evidencia_principal,
        'img_pil': img_pil
    }

print('Funcion de diagnostico combinado lista.')

In [ ]:
# ── Ejemplo integrado sobre una imagen del conjunto de prueba ─────────────

if os.path.exists(carpeta_test) and len(os.listdir(carpeta_test)) > 0:
    ejemplo_imagen = os.path.join(carpeta_test, sorted(os.listdir(carpeta_test))[0])
    resultado_integrado = diagnostico_combinado(
        ejemplo_imagen, clasificador, modelo_yolo_final,
        CLASES_CLASIFICACION, CLASES_YOLO
    )

    print('=' * 60)
    print('  EJEMPLO DE DIAGNOSTICO INTEGRADO')
    print('=' * 60)
    print(f'  Clasificacion general : {resultado_integrado["clase_general"]} '
          f'({resultado_integrado["confianza_clasificacion"]:.1f}%)')
    if resultado_integrado['evidencia_principal']:
        ev = resultado_integrado['evidencia_principal']
        print(f'  Objeto detectado YOLO : {ev["clase"]}')
        print(f'  Confianza YOLO        : {ev["confianza"]:.2f}')
    else:
        print('  No se detecto evidencia especifica adicional a la señal misma.')
    print('=' * 60)
else:
    print('AVISO: ejecutar primero el split y el entrenamiento para tener ejemplos de test.')

### 8.2 Ejemplo integrado documentado

```
Clasificacion general : deteriorada
Objeto detectado por YOLO : grafiti_o_pintura
Confianza YOLO : 0.89

Diagnostico:
La imagen muestra una señal de transito clasificada como deteriorada.
La evidencia visual confirma la presencia de grafiti que cubre parte
de la superficie reflectante de la señal, lo cual reduce su legibilidad
y puede confundir a los conductores.

Recomendacion:
Programar la limpieza o reemplazo de la señal y reportar el grafiti
a la unidad de mantenimiento vial correspondiente.
```

---
## 9. Diagnóstico textual combinado

In [ ]:
# ── Plantillas de diagnostico textual por clase YOLO ──────────────────────

PLANTILLAS_DIAGNOSTICO = {
    'grafiti_o_pintura': {
        'descripcion': 'grafiti o pintura no autorizada sobre la superficie de la señal',
        'recomendacion': 'Programar limpieza o reemplazo de la señal y reportar el grafiti a la unidad de mantenimiento vial.'
    },
    'oxido_o_corrosion': {
        'descripcion': 'oxido o corrosion en el material de la señal',
        'recomendacion': 'Evaluar el reemplazo de la placa o aplicar tratamiento anticorrosivo segun el grado de deterioro.'
    },
    'vegetacion_obstruyendo': {
        'descripcion': 'vegetacion (ramas u hojas) que obstruye parcialmente la señal',
        'recomendacion': 'Coordinar con el area de mantenimiento de parques y jardines la poda de la vegetacion cercana.'
    },
    None: {
        'descripcion': 'sin evidencia especifica adicional detectada por el modelo',
        'recomendacion': 'Mantener monitoreo periodico; no se identifico una causa visual puntual del estado reportado.'
    }
}

def generar_diagnostico_textual(resultado_integrado):
    """Genera el texto de diagnostico combinando clasificacion + deteccion YOLO."""
    clase_general = resultado_integrado['clase_general']
    evidencia = resultado_integrado['evidencia_principal']

    clase_yolo = evidencia['clase'] if evidencia else None
    confianza_yolo = evidencia['confianza'] if evidencia else None
    plantilla = PLANTILLAS_DIAGNOSTICO.get(clase_yolo, PLANTILLAS_DIAGNOSTICO[None])

    texto = []
    texto.append(f'La imagen fue clasificada como: {clase_general}.')
    texto.append('Evidencia visual detectada:')
    if evidencia:
        texto.append(f'  - Objeto o condicion: {clase_yolo}')
        texto.append(f'  - Confianza: {confianza_yolo:.2f}')
        texto.append(f'  - Ubicacion aproximada: {[round(c) for c in evidencia["bbox"]]}')
    else:
        texto.append('  - No se detecto un objeto o condicion especifica adicional a la señal.')
    texto.append('')
    texto.append(f'Diagnostico:')
    texto.append(f'La escena presenta una señal {clase_general} debido a la presencia de '
                 f'{plantilla["descripcion"]}.')
    texto.append('')
    texto.append(f'Recomendacion:')
    texto.append(plantilla['recomendacion'])

    return '\n'.join(texto)

# Ejemplo de uso (requiere haber ejecutado la celda de la Seccion 8)
if 'resultado_integrado' in globals():
    print(generar_diagnostico_textual(resultado_integrado))

### Plantilla general de diagnóstico

```
La imagen fue clasificada como: [clase general].
Evidencia visual detectada:
  - Objeto o condicion: [clase YOLO]
  - Confianza: [valor]
  - Ubicacion aproximada: [bounding box]

Diagnostico:
La escena presenta una señal [clase general] debido a la presencia
de [descripcion de la evidencia detectada].

Recomendacion:
[accion correctiva sugerida].
```

---
## 10. Proyección multimodal para el proyecto final

Para el proyecto final se propone incorporar un modelo **multimodal** que complemente la clasificación y la detección con una capa de interpretación en lenguaje natural, usando las mismas imágenes del proyecto.

### Opciones evaluadas

| Modelo | Rol propuesto | Aplicación concreta en el proyecto |
|---|---|---|
| **CLIP** | Comparar la imagen con descripciones textuales candidatas (clasificación zero-shot) | Validar de forma independiente si la imagen corresponde mejor a *"señal con grafiti"*, *"señal oxidada"* o *"señal obstruida por vegetación"* |
| **BLIP** | Generar automáticamente una descripción en lenguaje natural de la imagen | Producir una primera versión del diagnóstico textual sin depender únicamente de plantillas fijas |
| **VQA** | Responder preguntas puntuales sobre la imagen | Hacer preguntas dirigidas como *"¿hay vegetación cubriendo la señal?"* para confirmar o contrastar la salida de YOLO |
| **Vision Transformer (ViT)** | Comparar su clasificación y mapas de atención con los de la CNN/MobileNetV2 | Verificar si el modelo "mira" la señal y la evidencia detectada, o si se basa en el fondo de la imagen (explicabilidad) |

### Ejemplo conceptual con imágenes propias del proyecto

```
Pregunta VQA:
¿Hay vegetacion bloqueando la señal de transito?

Respuesta esperada:
Si, se observan ramas cubriendo parcialmente la mitad superior
de la señal de cruce peatonal.
```

```
Comparacion CLIP (zero-shot):
Imagen: senal_004.jpg
Texto candidato 1: "a traffic sign covered by vegetation"      -> similitud 0.81
Texto candidato 2: "a traffic sign with graffiti"               -> similitud 0.34
Texto candidato 3: "a clean and visible traffic sign"           -> similitud 0.22

Resultado: coincide con la deteccion de YOLO (vegetacion_obstruyendo)
```

Esta capa multimodal no reemplaza a YOLO ni al clasificador: actúa como una **verificación cruzada** y como generador de lenguaje natural más rico para el diagnóstico final.

---
## 11. Conclusiones del Avance 3

1. Se logró pasar de una clasificación general de la imagen completa a la **localización de evidencias visuales concretas** (la señal, el grafiti, el óxido, la vegetación) mediante YOLO.
2. Se definieron 4 clases de detección, todas correspondientes a objetos o condiciones físicamente observables, evitando conceptos abstractos como "riesgo alto".
3. El dataset YOLO reutiliza las 240 imágenes ya existentes, anotadas en Roboflow y divididas en 70% train / 20% val / 10% test.
4. Se entrenó un primer detector **YOLOv8n** (50 épocas, imgsz=640, batch=8) y se documentó el procedimiento completo para reportar precision, recall, mAP50 y mAP50-95.
5. Se construyó un pipeline de **integración** que combina la salida del clasificador MobileNetV2 (Avance 2) con la salida de YOLO (Avance 3) para generar un **diagnóstico textual** automático con recomendación de acción.
6. Se dejó planteada la incorporación de un modelo **multimodal** (CLIP/BLIP/VQA) como siguiente paso hacia el proyecto final del 12 de julio, para enriquecer el diagnóstico con lenguaje natural y verificación cruzada.

### Próximos pasos inmediatos
- Completar la anotación real en Roboflow y ejecutar el entrenamiento con GPU para obtener las métricas reales de detección.
- Ampliar el número de instancias por clase de evidencia (grafiti, óxido, vegetación) para mejorar el AP50 de esas categorías.
- Integrar el pipeline completo (clasificación + YOLO + diagnóstico) en un único script o función reutilizable para el proyecto final.
- Evaluar de forma exploratoria CLIP o BLIP sobre el mismo conjunto de imágenes de prueba.
